# NeuroGraph-X · Week 6 · Final models, frozen set measured once, calibration
Owner: Laiba Afridi. Cards #96, #97, #98 (memo 0010, memo 0011 section 5).

For each atlas this notebook trains **one** final model on all 761 development participants with the configuration chosen in `docs/reports/tuning_selection.md`, measures it **once** on the frozen 100, registers it, then fits its calibrator on the CV run's out-of-fold predictions (never on the frozen set).

| Atlas | Selected CV run | Frozen measurement |
|---|---|---|
| AAL | `d3611e1d` | first |
| CC200 | `c2516a5f` | **declared repeat** (`00252f2b` stays in the log) |
| Harvard-Oxford | `d1873b17` | first |
| Dosenbach 160 | `f9f7e584` | first |

**Run it once, without watching:** add the `neurographx-code` dataset (version made from the reviewed Week 6 branch), set GPU T4 and Internet On, then **Save Version → Save & Run All (Commit)**. About 1 hour 15 minutes, almost all of it the connectome rebuild. Every cell checks itself and stops the run if something is wrong; a stopped run has measured nothing for the atlases after the stop.

**Never edit a cell after it has measured an atlas and run again with other settings.** Re-running the same notebook is safe: an atlas already measured with its selected CV run is skipped, not measured again.

## 1. Copy the code into a writable folder

In [ ]:
import glob, json, os, shutil, subprocess, sys
# The input can hold more than one copy of the code (a dataset version keeps earlier files
# unless they are deleted), so pick the Week 6 copy explicitly and stop at once otherwise.
found = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/pyproject.toml", recursive=True))
print("code folders in the input:", found)
week6 = [d for d in found if os.path.exists(f"{d}/tests/test_calibration.py")
         and os.path.exists(f"{d}/notebooks/neurographx-w6-final-models.ipynb")]
assert len(week6) == 1, f"expected exactly one Week 6 code folder, found {len(week6)}: check the dataset version"
code_root = week6[0]
print("using:", code_root)
REPO = "/kaggle/working/ngx"
if not os.path.exists(REPO):
    shutil.copytree(code_root, REPO)
os.chdir(REPO)
print("repo:", REPO, "| commit:", open("COMMIT").read().strip() if os.path.exists("COMMIT") else "unknown")
ENV = {**os.environ, "PYTHONPATH": "src", "PYTHONUNBUFFERED": "1"}

def run(*cmd):
    """Run a command, show its output as it comes, stop the notebook if it fails."""
    print("$", " ".join(cmd), flush=True)
    done = subprocess.run(list(cmd), env=ENV)
    assert done.returncode == 0, f"failed with exit code {done.returncode}: {' '.join(cmd)}"

def runs():
    return [json.loads(l) for l in open("runs/run_log.jsonl") if l.strip()]

In [ ]:
!pip install -q torch_geometric==2.8.0.post1 nilearn==0.14.1 tabulate
import torch, torch_geometric; print("torch", torch.__version__, "| PyG", torch_geometric.__version__, "| GPU", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU: set Accelerator to GPU T4"

## 2. Rebuild the connectomes (memo 0007). The download is resumable; about an hour.

In [ ]:
run(sys.executable, "scripts/01_fetch_phenotypic.py")
for attempt in range(3):  # the download can drop; each retry continues where it stopped
    if subprocess.run([sys.executable, "scripts/04_fetch_all_atlases.py", "--quiet"], env=ENV).returncode == 0:
        break
else:
    raise SystemExit("04_fetch_all_atlases.py failed three times")
run(sys.executable, "scripts/07_build_connectomes.py")
run(sys.executable, "scripts/08_verify_connectomes.py")

## 3. Check: the rebuilt matrices must match the committed report before any training

In [ ]:
expected = {  # key: (participants, nodes, flat-region participants, NaN edges), docs/reports/connectome_summary.md
    "rois_aal": (861, 116, 54, 2014), "rois_cc200": (861, 200, 15, 3969),
    "rois_ho": (861, 111, 198, 1055), "rois_dosenbach160": (861, 161, 46, 2440)}
rows = {}
for line in open("docs/reports/connectome_summary.md"):
    cells = [c.strip() for c in line.strip().strip("|").split("|")]
    if len(cells) > 6 and cells[1] in expected:
        rows[cells[1]] = (int(cells[2]), int(cells[3]), int(cells[5]), int(cells[6]))
for key, want in expected.items():
    print(f"{key:>18}: {rows.get(key)}  expected {want}")
    assert rows.get(key) == want, f"{key} differs from the committed report: stop and ask"
print("All four atlases match the committed report.")

## 4. Tests (all must pass)

In [ ]:
run(sys.executable, "-m", "pytest", "-q", "tests/test_leakage.py", "tests/test_gnn_data.py",
    "tests/test_run_logger.py", "tests/test_inference.py", "tests/test_oof.py", "tests/test_tuning.py",
    "tests/test_calibration.py")

## 5. Check before measuring: the selection is reproduced from the committed run log
The selection script is re-run here from the log alone (it never reads the frozen set). If it does not choose exactly the four runs in the table above, nothing is measured.

In [ ]:
SELECTED = {"aal": "d3611e1d", "cc200": "c2516a5f", "ho": "d1873b17", "dosenbach160": "f9f7e584"}
KEYS = {"aal": "rois_aal", "cc200": "rois_cc200", "ho": "rois_ho", "dosenbach160": "rois_dosenbach160"}
REPEAT_REASON = "post-tuning measurement per memo 0011"

out = subprocess.run([sys.executable, "scripts/select_config.py", "--out", "/tmp/tuning_selection_check.md"],
                     env=ENV, capture_output=True, text=True)
print(out.stdout, out.stderr)
assert out.returncode == 0
for atlas, rid in SELECTED.items():
    assert f"(run {rid})" in out.stdout, f"{atlas}: the selection is not {rid}: stop and ask"
earlier = {a: [r["run_id"] for r in runs() if r.get("stage") == "final" and r["atlas"] == a
               and r["metrics"].get("cv_run_id") != SELECTED[a]] for a in SELECTED}
print("final runs measured before this notebook:", earlier)
assert earlier == {"aal": [], "cc200": ["00252f2b"], "ho": [], "dosenbach160": []}, \
    "the run log does not hold exactly one earlier final run (CC200 00252f2b): stop and ask"
print("Selection reproduced; only CC200 has an earlier frozen measurement. Safe to measure.")

## 6. Final models: one per atlas, the frozen set measured once each
Each cell trains on all 761 development participants for the median best epoch of the selected CV run (density, hidden size and class weights are copied from that run by `train_gnn.py`), then predicts the frozen 100 once, logs the run with the checkpoint's SHA-256, saves the frozen predictions and registers the model. CC200 is the declared repeat.

In [ ]:
EVIDENCE = "/kaggle/working/evidence"

def save_evidence():
    """Copy what exists so far, so a later failure cannot lose an earlier measurement."""
    for sub in ("runs/frozen_predictions", "runs/calibration", "checkpoints", "figures"):
        os.makedirs(f"{EVIDENCE}/{sub}", exist_ok=True)
    shutil.copy("runs/run_log.jsonl", EVIDENCE)
    if os.path.exists("runs/model_registry.json"):
        shutil.copy("runs/model_registry.json", EVIDENCE)
    for pattern, sub in (("runs/frozen_predictions/*.csv", "runs/frozen_predictions"),
                         ("runs/calibration/*.json", "runs/calibration"),
                         ("checkpoints/*.pt", "checkpoints"),
                         ("docs/reports/figures/calibration_*.png", "figures")):
        for f in glob.glob(pattern):
            shutil.copy(f, f"{EVIDENCE}/{sub}/")

def final(atlas):
    done = [r for r in runs() if r.get("stage") == "final" and r["atlas"] == atlas
            and r["metrics"].get("cv_run_id") == SELECTED[atlas]]
    if done:
        print(f"{atlas}: already measured in this session (run {done[-1]['run_id']}); not measured again")
        return done[-1]["run_id"]
    cmd = [sys.executable, "scripts/train_gnn.py", "--stage", "final", "--atlas", KEYS[atlas],
           "--epochs-from-cv", SELECTED[atlas]]
    if atlas == "cc200":
        cmd += ["--allow-repeat", "--repeat-reason", REPEAT_REASON]
    run(*cmd)
    rid = runs()[-1]["run_id"]
    save_evidence()
    return rid

FINAL = {}

In [ ]:
FINAL["aal"] = final("aal")

In [ ]:
FINAL["cc200"] = final("cc200")

In [ ]:
FINAL["ho"] = final("ho")

In [ ]:
FINAL["dosenbach160"] = final("dosenbach160")

## 7. Calibration (memo 0010): fitted on each selected CV run's OOF table, scored on the frozen predictions already saved
No model is run again and nothing is fitted on the frozen set.

In [ ]:
todo = [rid for atlas, rid in FINAL.items() if not os.path.exists(f"runs/calibration/{atlas}_{rid}.json")]
if todo:
    run(sys.executable, "scripts/calibrate.py", "--run", *todo)
save_evidence()

## 8. Results and checks

In [ ]:
by_id = {r["run_id"]: r for r in runs()}
registry = json.load(open("runs/model_registry.json"))
print(f"{'atlas':>13} {'run':>9} {'CV bal':>7} {'acc':>6} {'bal':>6} {'AUC':>6} {'base':>6}  {'ECE before->after (frozen)'}")
for atlas, rid in FINAL.items():
    r, cv = by_id[rid], by_id[SELECTED[atlas]]
    f = r["metrics"]["frozen"]
    cal = json.load(open(f"runs/calibration/{atlas}_{rid}.json"))
    print(f"{atlas:>13} {rid:>9} {cv['metrics']['cv_balanced_accuracy']['mean']:>7.4f} {f['accuracy']:>6.3f} "
          f"{f['balanced_accuracy']:>6.3f} {f['auc']:>6.3f} {f['majority_baseline']:>6.2f}  "
          f"{cal['frozen']['before']['ece']} -> {cal['frozen']['after']['ece']}")
    assert r["reload_identical"], f"{atlas}: reloaded checkpoint gave different predictions"
    assert r.get("checkpoint_sha256"), f"{atlas}: no checkpoint hash in the log"
    entry = registry[atlas] if isinstance(registry, dict) and atlas in registry else None
    assert entry is None or entry["run_id"] == rid, f"{atlas}: registry does not serve run {rid}"
cc = by_id[FINAL["cc200"]]
assert cc.get("repeat_reason") == REPEAT_REASON, "CC200 repeat reason missing from the log"
print("\nAll four final runs logged, checkpoints reload identically, CC200 repeat declared.")

## 9. Evidence: one small zip to download

In [ ]:
save_evidence()
shutil.make_archive("/kaggle/working/w6_results", "zip", EVIDENCE)
print(sum(1 for _ in open(f"{EVIDENCE}/run_log.jsonl")), "lines in the run log")
for sub in ("checkpoints", "runs/frozen_predictions", "runs/calibration", "figures"):
    print(f"{sub}: {sorted(os.listdir(f'{EVIDENCE}/{sub}'))}")
print("zip:", round(os.path.getsize("/kaggle/working/w6_results.zip") / 1e6, 2), "MB")

Download **`w6_results.zip`** from the version's Output (Output → `/kaggle/working`). It holds the run log (64 lines), the registry, the four new checkpoints, the four frozen-prediction files, the four calibration files and their reliability figures.